In [2]:
import pandas as pd
import geopandas as gpd

In [3]:
trees = gpd.read_file("../data/processed/usda_philly_trees.geojson")

In [4]:
usda_misses = trees.loc[trees["plant_code"].isna()]

In [5]:
usda_misses.groupby("scientific_name", as_index=False).size().sort_values(ascending=False, by="size")

,scientific_name,size
62,prunus species,7458
80,syringa reticulata,3772
45,malus species,2758
84,tilia species,2681
61,prunus serrulata,2546
...,...,...
5,acer henryii,2
40,kalopanax semtemlobus,1
51,populus trichocarpa,1
52,prunus amanogawa,1


In [6]:
# some trees still have "species" meaning they are not specific
# will remove to see how many specifc species are missing
specific_trees = trees.loc[trees["Species"] != "species"]
print((len(specific_trees.loc[~specific_trees["plant_code"].isna()])) / len(trees))

0.688685973272903


In [7]:
specific_misses = specific_trees.loc[specific_trees["plant_code"].isna()]
specific_misses.groupby("scientific_name", as_index=False).size().sort_values(ascending=False, by="size").head(40)

,scientific_name,size
55,syringa reticulata,3772
42,prunus serrulata,2546
1,acer campestre,2519
51,sophora japonica,1833
43,prunus subhirtella,1183
28,maackia amurensis,1031
13,carpinus betulus,1025
41,prunus sargentii,951
6,acer tataricum,782
44,prunus triloba,742


19% of tree species are missing from the USDA database.

In [8]:
mobot = pd.read_csv("/Users/prince/philly-tree-mapper/notebooks/mobot_plant_data.csv")

In [9]:
mobot.groupby("Status", as_index=False).size().sort_values(by="size")

,Status,size
0,"fuzzy (sci=0.63, com=1.00)",1
1,"fuzzy (sci=0.73, com=1.00)",1
2,"fuzzy (sci=0.74, com=1.00)",1
3,"fuzzy (sci=0.76, com=1.00)",1
4,"fuzzy (sci=0.78, com=0.96)",1
5,"fuzzy (sci=0.78, com=1.00)",1
6,manual search,2
10,single result,2
8,no exact match,28
9,no results container,41


In [10]:
# merge mobot and philly trees
mobot_trees = pd.merge(trees, mobot, left_on="scientific_name", right_on="Scientific Name", how="left")

In [11]:
mobot_trees

,objectid,tree_name,scientific_name,common_name,Genus,Species,plant_code,Morphology/Physiology | Active Growth Period,Morphology/Physiology | Fall Conspicuous,Morphology/Physiology | Flower Color,...,Reproduction | Fruit/Seed Period End,Suitability/Use | Palatable Human,geometry,Scientific Name,Bloom Time,Bloom Description,Flower,Leaf,Status,Detail URL
0,15194,Abies balsamea - balsam fir,abies balsamea,balsam fir,Abies,balsamea,ABBA,Spring and Summer,No,Yellow,...,Fall,No,POINT (-75.1538 40.02521),abies balsamea,Non-flowering,Non-flowering,N/a,"Fragrant, Evergreen",matched,https://www.missouribotanicalgarden.org/PlantF...
1,30419,Abies balsamea - balsam fir,abies balsamea,balsam fir,Abies,balsamea,ABBA,Spring and Summer,No,Yellow,...,Fall,No,POINT (-75.14772 39.94421),abies balsamea,Non-flowering,Non-flowering,N/a,"Fragrant, Evergreen",matched,https://www.missouribotanicalgarden.org/PlantF...
2,36114,Abies balsamea - balsam fir,abies balsamea,balsam fir,Abies,balsamea,ABBA,Spring and Summer,No,Yellow,...,Fall,No,POINT (-75.21361 39.93275),abies balsamea,Non-flowering,Non-flowering,N/a,"Fragrant, Evergreen",matched,https://www.missouribotanicalgarden.org/PlantF...
3,16196,Abies balsamea - balsam fir,abies balsamea,balsam fir,Abies,balsamea,ABBA,Spring and Summer,No,Yellow,...,Fall,No,POINT (-75.11614 40.04005),abies balsamea,Non-flowering,Non-flowering,N/a,"Fragrant, Evergreen",matched,https://www.missouribotanicalgarden.org/PlantF...
4,25871,Abies balsamea - balsam fir,abies balsamea,balsam fir,Abies,balsamea,ABBA,Spring and Summer,No,Yellow,...,Fall,No,POINT (-75.21198 39.98109),abies balsamea,Non-flowering,Non-flowering,N/a,"Fragrant, Evergreen",matched,https://www.missouribotanicalgarden.org/PlantF...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150030,151710,Prunus subhirtella - snow goose cherry,prunus subhirtella,snow goose cherry,Prunus,subhirtella,NaN,NaN,NaN,NaN,...,NaN,NaN,POINT (-75.10246 39.98252),prunus subhirtella,April,Pink to white,Showy,N/a,matched,https://www.missouribotanicalgarden.org/PlantF...
150031,151711,Prunus subhirtella - snow goose cherry,prunus subhirtella,snow goose cherry,Prunus,subhirtella,NaN,NaN,NaN,NaN,...,NaN,NaN,POINT (-75.10213 39.98269),prunus subhirtella,April,Pink to white,Showy,N/a,matched,https://www.missouribotanicalgarden.org/PlantF...
150032,151712,Syringia reticulata - ivory silk japanese tree...,syringia reticulata,ivory silk japanese tree lilac,Syringia,reticulata,NaN,NaN,NaN,NaN,...,NaN,NaN,POINT (-75.10201 39.98276),syringia reticulata,N/a,N/a,N/a,N/a,no results container,NaN
150033,151713,Prunus cerasifera - purpleleaf plum,prunus cerasifera,purpleleaf plum,Prunus,cerasifera,NaN,NaN,NaN,NaN,...,NaN,NaN,POINT (-75.05639 40.01616),prunus cerasifera,April,White,"Showy, Fragrant",N/a,matched,https://www.missouribotanicalgarden.org/PlantF...


In [12]:
print(len(mobot_trees.loc[mobot_trees["Status"].isna()]))

0


"No results container" in mobot status column implies a miss. will use that to find how many trees miss

In [13]:
mobot_matches = mobot_trees.loc[mobot_trees["Status"] != "no results container"]
mobot_misses = mobot_trees.loc[mobot_trees["Status"] == "no results container"]
print(len(mobot_matches) / len(trees))


0.9775985603359216


In [16]:
philly_species_pheno = mobot_trees.drop_duplicates(subset="tree_name")
philly_species_pheno.to_file("/Users/prince/philly-tree-mapper/data/processed/philly_species_pheno.geojson", driver="GeoJSON")